# Mengunduh Data CH4 di Kabupaten Gresik

Notebook ini mengambil data konsentrasi **CH4 (Metana)** dari Sentinel-5P melalui openEO, lalu menyimpannya sebagai file netCDF (`.nc`) dan CSV (`.csv`).

In [7]:
import openeo

In [8]:
connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()

Authenticated using refresh token.


## Area of Interest (AOI)

Polygon lokasi pengamatan di Kabupaten Gresik. Format koordinat: `[longitude, latitude]`.

In [9]:
aoi = {
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "properties": {},
      "geometry": {
        "type": "Polygon",
        "coordinates": [[
          [112.6193968, -7.1514786],
          [112.6600158, -7.1514786],
          [112.6600158, -7.1927923],
          [112.619805,  -7.1927923],
          [112.6193968, -7.1514786]
        ]]
      }
    }
  ]
}

## Load Data CH4

In [10]:
s5 = connection.load_collection(
    "SENTINEL_5P_L2",
    temporal_extent=["2025-08-24", "2026-08-24"],
    spatial_extent={
        "west": 112.6193968,
        "south": -7.1927923,
        "east": 112.6600158,
        "north": -7.1514786,
    },
    bands=["CH4"],
)

In [11]:
# Rata-rata harian, lalu rata-rata di dalam polygon AOI
s5 = s5.aggregate_temporal_period(reducer="mean", period="day")
s5 = s5.aggregate_spatial(reducer="mean", geometries=aoi)

## Jalankan Batch Job

In [12]:
job = s5.execute_batch(title="CH4 Gresik", outputfile="../data/nc/polutan_CH4_gresik.nc")

0:00:00 Job 'j-26082811584243d4a227cccf6350621a': send 'start'
0:00:02 Job 'j-26082811584243d4a227cccf6350621a': queued (progress 0%)
0:00:08 Job 'j-26082811584243d4a227cccf6350621a': queued (progress 0%)
0:00:14 Job 'j-26082811584243d4a227cccf6350621a': queued (progress 0%)
0:00:22 Job 'j-26082811584243d4a227cccf6350621a': queued (progress 0%)
0:00:33 Job 'j-26082811584243d4a227cccf6350621a': queued (progress 0%)
0:00:45 Job 'j-26082811584243d4a227cccf6350621a': queued (progress 0%)
0:01:01 Job 'j-26082811584243d4a227cccf6350621a': queued (progress 0%)
0:01:20 Job 'j-26082811584243d4a227cccf6350621a': queued (progress 0%)
0:01:44 Job 'j-26082811584243d4a227cccf6350621a': running (progress N/A)
0:02:14 Job 'j-26082811584243d4a227cccf6350621a': running (progress N/A)
0:02:52 Job 'j-26082811584243d4a227cccf6350621a': finished (progress 100%)


## Konversi netCDF ke CSV

In [13]:
import netCDF4
import pandas as pd

ds = netCDF4.Dataset("../data/nc/polutan_CH4_gresik.nc")

ch4 = ds.variables["CH4"][0, :]   # feature 0, semua waktu
time = ds.variables["t"][:]
time_units = ds.variables["t"].units
dates = netCDF4.num2date(time, units=time_units)

# Deret tanggal penuh 1 tahun
full_dates = pd.date_range(start="2025-08-24", end="2026-08-23", freq="D")
ch4_map = {d.strftime("%Y-%m-%d"): float(v) for d, v in zip(dates, ch4)}

df = pd.DataFrame({"date": full_dates.strftime("%Y-%m-%d")})
df["CH4"] = df["date"].map(ch4_map)

df.to_csv("../data/csv/CH4_gresik_timeseries.csv", index=False)
print("CSV disimpan:", len(df), "baris")

CSV disimpan: 365 baris
